In [6]:
from pathlib import Path
import cv2
import json
from tqdm.auto import tqdm

# Đường dẫn video đầu vào
VIDEO_PATH = Path(r"../resources/videos/video00.mp4")

# Thư mục lưu các frame
OUTPUT_DIR = Path(r"../resources/image_train")

# Lấy một frame sau mỗi N frame gốc
FRAME_INTERVAL = 30

# Định dạng ảnh: jpg hoặc png
IMAGE_FORMAT = "jpg"

# JPEG quality: 95 = chất lượng cao
JPEG_QUALITY = 95

# True: ghi đè các frame cùng tên nếu đã tồn tại
OVERWRITE = False

In [7]:
def extract_frames(
    video_path,
    output_dir,
    frame_interval=2,
    image_format="jpg",
    jpeg_quality=95,
    overwrite=False,
):
    video_path = Path(video_path)
    output_dir = Path(output_dir)

    if not video_path.is_file():
        raise FileNotFoundError(f"Không tìm thấy video: {video_path}")

    if frame_interval < 1:
        raise ValueError("frame_interval phải >= 1")

    image_format = image_format.lower().lstrip(".")
    if image_format not in {"jpg", "png"}:
        raise ValueError("image_format chỉ hỗ trợ jpg hoặc png")

    output_dir.mkdir(parents=True, exist_ok=True)

    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        raise RuntimeError(f"Không mở được video: {video_path}")

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    saved = 0
    failed = 0
    frame_idx = 0

    print(f"Video: {video_path.name}")
    print(f"Resolution: {width}x{height}")
    print(f"FPS: {fps:.2f}")
    print(f"Total frames: {total_frames}")
    print(f"Interval: {frame_interval}")

    try:
        with tqdm(total=total_frames, desc="Extracting") as pbar:
            while True:
                ret, frame = cap.read()
                if not ret:
                    break

                # frame_idx bắt đầu từ 0
                if frame_idx % frame_interval == 0:
                    output_path = output_dir / (
                        f"frame_{frame_idx:08d}.{image_format}"
                    )

                    if output_path.exists() and not overwrite:
                        saved += 1
                    else:
                        params = []
                        if image_format == "jpg":
                            params = [
                                cv2.IMWRITE_JPEG_QUALITY,
                                jpeg_quality,
                            ]

                        ok = cv2.imwrite(
                            str(output_path),
                            frame,
                            params,
                        )

                        if ok:
                            saved += 1
                        else:
                            failed += 1

                frame_idx += 1
                pbar.update(1)

    finally:
        cap.release()

    metadata = {
        "video": str(video_path.resolve()),
        "fps": fps,
        "width": width,
        "height": height,
        "total_frames": frame_idx,
        "frame_interval": frame_interval,
        "saved_frames": saved,
        "failed_frames": failed,
        "image_format": image_format,
    }

    with open(
        output_dir / "metadata.json",
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(metadata, f, indent=2, ensure_ascii=False)

    print("\nHoàn tất!")
    print(f"Frames: {output_dir}")
    print(f"Saved: {saved}")
    print(f"Failed: {failed}")

    return metadata

In [8]:
metadata = extract_frames(
    video_path=VIDEO_PATH,
    output_dir=OUTPUT_DIR,
    frame_interval=FRAME_INTERVAL,
    image_format=IMAGE_FORMAT,
    jpeg_quality=JPEG_QUALITY,
    overwrite=OVERWRITE,
)

Video: video00.mp4
Resolution: 1920x1080
FPS: 59.94
Total frames: 1193
Interval: 30


Extracting:   0%|          | 0/1193 [00:00<?, ?it/s]

Extracting: 100%|██████████| 1193/1193 [00:09<00:00, 121.01it/s]



Hoàn tất!
Frames: ..\resources\image_train
Saved: 40
Failed: 0
